# Adaptive Cyber-Physical Security: AI Network Anomaly Detection
## Exploratory Data Analysis & Unsupervised Modeling Notebook

This notebook demonstrates the end-to-end unsupervised network anomaly detection pipeline on the **NSL-KDD** benchmark dataset:
1. **Data Ingestion & Telemetry Inspection**
2. **Data Preprocessing with Scikit-Learn Pipelines**
3. **PCA Dimensionality Reduction & Variance Spectrum**
4. **Isolation Forest Unsupervised Anomaly Modeling**
5. **Benchmark Evaluation & Attack Class Diagnostics**

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project modules are accessible
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.data_loader import load_config, load_dataset, get_dataset_summary
from src.preprocessing import TrafficPreprocessor
from src.feature_engineering import PCAManager
from src.detector import IsolationForestDetector
from src.evaluation import evaluate_predictions, compare_configurations

config = load_config(root_dir / 'config' / 'config.yaml')
print('Configured Dataset:', config['dataset']['name'])

### 1. Ingest Dataset & Telemetry Inspection

In [ ]:
train_path = root_dir / config['dataset']['paths']['train_file']
test_path = root_dir / config['dataset']['paths']['test_file']

X_train, y_train, cat_train = load_dataset(train_path, config=config)
X_test, y_test, cat_test = load_dataset(test_path, config=config)

print(f'Train Features Shape: {X_train.shape}')
print(f'Held-Out Test Shape: {X_test.shape}')
print('\nTraining Label Distribution:')
print(y_train.value_counts(normalize=True).rename({0: 'Normal (0)', 1: 'Attack (1)'}))

### 2. Preprocessing & Leakage-Free Transformation

In [ ]:
# Preprocessor fitted strictly on training data
preprocessor = TrafficPreprocessor(
    categorical_features=config['schema']['categorical_features'],
    binary_features=config['schema']['binary_features']
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f'Transformed feature dimension: {X_train_proc.shape[1]} features (One-Hot Encoded + Scaled)')

### 3. PCA Dimensionality Reduction

In [ ]:
pca = PCAManager(random_state=42)
pca.fit(X_train_proc, variance_threshold=0.90)
summary = pca.get_variance_summary()

print(f'Components explaining 90% variance: {summary["n_components_90_variance"]}')
print(f'2D Latent Space Variance Explained: {summary["total_2d_explained_variance"]:.2%}')

# Plot Cumulative Explained Variance
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(summary['cumulative_variance_ratios']) + 1), summary['cumulative_variance_ratios'], marker='o', color='#0284c7')
plt.axhline(0.90, color='r', linestyle='--', label='90% Threshold')
plt.title('PCA Cumulative Explained Variance Spectrum')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Variance Ratio')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 4. Unsupervised Isolation Forest Modeling

In [ ]:
# Model trained without ground-truth labels
detector = IsolationForestDetector(n_estimators=150, max_samples=512, contamination=0.10, random_state=42)
# Fit on benign normal baseline
normal_mask = (y_train == 0)
detector.fit(X_train_proc[normal_mask])

# Evaluate on held-out test data
y_test_pred = detector.predict(X_test_proc)
scores_test = detector.predict_anomaly_scores(X_test_proc)

eval_results = evaluate_predictions(y_test.to_numpy(), y_test_pred, anomaly_scores=scores_test, attack_categories=cat_test)

print('Held-out Benchmark Evaluation (KDDTest+):')
print(f'Precision: {eval_results["precision"]:.4f}')
print(f'Recall:    {eval_results["recall"]:.4f}')
print(f'F1-Score:  {eval_results["f1_score"]:.4f}')
print(f'False Alarm Rate (FPR): {eval_results["false_positive_rate"]:.2%}')
print(f'PR-AUC:    {eval_results["pr_curve"]["average_precision"]:.4f}')

### 5. Category-Specific Attack Recall Breakdown

In [ ]:
cat_df = pd.DataFrame(eval_results['per_category_metrics']).T
print(cat_df)